# Pipetly — API Clients & Orchestrator Test Notebook

Use this notebook to interactively run searches against any combination of the
configured API clients and inspect the raw papers returned before filtering.

**Sections**
1. Setup & imports
2. Configuration — set your query here
3. Individual client search (EuropePMC · SemanticScholar · Elsevier · CrossRef · OpenAlex · Scopus · PMC · CORE)
4. Deduplication — combine & deduplicate all search results by DOI
5. Individual client full-text retrieval (EuropePMC · PMC · Elsevier · Semantic Scholar · Unpaywall · CORE)
6. Parse & display results
7. Analysis (counts by source, full-text availability, year distribution)

## 1 · Setup & Imports

In [1]:
import sys, os, asyncio, logging, warnings
from pathlib import Path

# ── Make sure the project root is on the path ────────────────────────────────
ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

# ── Standard imports ─────────────────────────────────────────────────────────
import pandas as pd
import json
warnings.filterwarnings("ignore")

# ── Logging – show INFO from Pipetly modules ─────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s – %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("notebook")

# ── Pipetly imports ───────────────────────────────────────────────────────────
from config import get_settings
from models.paper import Paper, FullText
from models.query import ExpandedQuery
from api_clients import (
    EuropePMCClient,
    SemanticScholarClient,
    ElsevierClient,
    CrossRefClient,
    OpenAlexClient,
    ScopusClient,
    PMCClient,
    UnpaywallClient,
    COREClient,
)
from processors.orchestrator import MultiSourceOrchestrator

settings = get_settings()
print(f"Model            : {settings.gemini_model}")
print(f"Max/source       : {settings.max_papers_per_source}")
print(f"Elsevier key set : {bool(settings.elsevier_api_key)}")
print(f"S2 key set       : {bool(settings.semantic_scholar_api_key)}")
print(f"Unpaywall email  : {bool(settings.unpaywall_email)}")
print(f"CORE key set     : {bool(settings.core_api_key)}")


Project root: C:\Users\rebec\OneDrive\Documentos\TFM\repo\Pipetly
Model            : google/gemini-2.5-flash
Max/source       : 10
Elsevier key set : True
S2 key set       : True
Unpaywall email  : True
CORE key set     : True


## 2 · Configuration

Edit the cell below to set your search query and per-source limit.

In [2]:
# ── Edit these ────────────────────────────────────────────────────────────────
QUERY = "FmtA structure"   # ← change to any search string
MAX_PER_SOURCE = 10                      # papers to fetch per API call

# ── Helper: convert a list of Paper objects to a tidy DataFrame ──────────────
def papers_to_df(papers: list[Paper]) -> pd.DataFrame:
    rows = []
    for p in papers:
        rows.append({
            "title":          p.title,
            "doi":            p.doi or "",
            "year":           p.year,
            "source":         p.source,
            "authors":        "; ".join(p.authors[:3]) + (" et al." if len(p.authors) > 3 else ""),
            "has_full_text":  p.full_text is not None,
            "abstract_only":  getattr(p.full_text, "is_abstract_only", None),
            "url":            p.url or "",
            "abstract":       (p.abstract or "")[:200],
        })
    return pd.DataFrame(rows)

print("Config ready. QUERY =", repr(QUERY))

Config ready. QUERY = 'FmtA structure'


## 3 · Individual Client Search

Run each subsection independently to inspect what a single API returns.

### 3a · Europe PMC

In [3]:
async with EuropePMCClient() as client:
    epmc_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"EuropePMC returned {len(epmc_papers)} papers")
papers_to_df(epmc_papers)

01:10:34 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/search?query=FmtA+structure&format=json&pageSize=10&resultType=core "HTTP/1.1 200 OK"


EuropePMC returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,The &lt;i&gt;Staphylococcus aureus&lt;/i&gt; e...,10.1128/mbio.02337-25,2025,europe_pmc,Berry KA; Verhoef MTA; Zheng Z et al.,False,None,https://europepmc.org/article/PMC/PMC12505958,<i>Staphylococcus aureus</i> teichoic acids ar...
1,Enhanced resistance of metal sequestering agen...,10.1038/s44259-025-00131-1,2025,europe_pmc,Paterson JR; Wadsworth JM; Lee RJ et al.,False,None,https://europepmc.org/article/PMC/PMC12229560,Chelators possess antibacterial properties lin...
2,"Structural, CSD, Molecular Docking, Molecular ...",10.1021/acsomega.4c06520,2025,europe_pmc,Mohandas P; Abdul Salam AA; Shenoy TN et al.,False,None,https://europepmc.org/article/PMC/PMC11822514,"1,2,4-Oxadiazoles are well recognized for thei..."
3,Gallium Resistance in &lt;i&gt;Staphylococcus ...,10.3390/applmicrobiol5010032,2025,europe_pmc,Ewunkem A; Simpson F; Holland D et al.,False,None,https://europepmc.org/article/PMC/PMC12393781,<h4>Background and objectives</h4>The imminent...
4,Clinical Isolate of &lt;i&gt;Candida tropicali...,10.4236/ojmm.2025.151002,2025,europe_pmc,Ewunkem A; Merrills L; Williams Z et al.,False,None,https://europepmc.org/article/PMC/PMC12094522,"In North Carolina, candida infections are on t..."
5,Drug-Repurposing Approach To Combat <i>Staphyl...,10.1021/acsomega.2c03671,2022,europe_pmc,Singh V; Dhankhar P; Dalal V et al.,False,None,https://europepmc.org/article/PMC/PMC9631409,<i>Staphylococcus aureus</i> is considered as ...
6,Exploring the inhibition mechanisms of momordi...,10.1038/s41598-025-24255-6,2025,europe_pmc,Yang Y; Li X; Hou P et al.,False,None,https://europepmc.org/article/PMC/PMC12594824,Serine/threonine phosphatase (Stp1) modulates ...
7,Discovery of potent anti-MRSA components from ...,10.1016/j.jpha.2024.01.006,2024,europe_pmc,Wu J; Hassan SSU; Zhang X et al.,False,None,https://europepmc.org/article/PMC/PMC11381741,Image 1.
8,Effect of Heat Input on Microstructural Evolut...,10.3390/ma18051148,2025,europe_pmc,Liu Y; Ma H; Wang Z et al.,False,None,https://europepmc.org/article/PMC/PMC11901722,In order to find the optimal heat input for si...
9,Structure and function of prodrug-activating p...,10.1016/j.biochi.2022.07.019,2023,europe_pmc,Velilla JA; Kenney GE; Gaudet R,False,None,https://europepmc.org/article/PMC/PMC10030199,Bacteria protect themselves from the toxicity ...


### 3b · Semantic Scholar

In [4]:
async with SemanticScholarClient() as client:
    s2_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"Semantic Scholar returned {len(s2_papers)} papers")
papers_to_df(s2_papers)

01:10:37 [INFO] httpx – HTTP Request: GET https://api.semanticscholar.org/graph/v1/paper/search?query=FmtA+structure&limit=10&fields=paperId%2CexternalIds%2Ctitle%2Cabstract%2Cyear%2Cauthors%2CopenAccessPdf "HTTP/1.1 200 OK"


Semantic Scholar returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,Structure-Based Identification of Potential Dr...,10.1007/s10930-020-09953-6,2021,semantic_scholar,V. Dalal; P. Dhankhar; Vishakha Singh et al.,False,None,,
1,Repurposing an Ancient Protein Core Structure:...,10.1016/j.jmb.2019.06.019,2019,semantic_scholar,V. Dalal; Pramod Kumar; G. Rakhaminov et al.,False,None,,FmtA is a penicillin-recognizing protein (PRP)...
2,Crystal Structure of FmtA from Staphylococcus ...,10.2210/PDB5ZH8/PDB,2019,semantic_scholar,V. Dalal; Pravindra Kumar; G. Rakhaminov et al.,False,None,,Abstract FmtA is a penicillin-recognizing prot...
3,Structure of FmtA-like protein,10.2210/pdb4gdn/pdb,2012,semantic_scholar,A. Cougnoux; L. Gibold; J. Delmas et al.,False,None,,
4,Analysis of structure-function relationships i...,10.1016/j.jmb.2012.09.017,2012,semantic_scholar,A. Cougnoux; L. Gibold; F. Robin et al.,False,None,,
5,In-silico functional and structural annotation...,10.1016/j.jmgm.2022.108262,2022,semantic_scholar,Vishakha Singh; P. Dhankhar; V. Dalal et al.,False,None,,Klebsiella pneumonia is known to cause several...
6,Drug-Repurposing Approach To Combat Staphyloco...,10.1021/acsomega.2c03671,2022,semantic_scholar,Vishakha Singh; P. Dhankhar; V. Dalal et al.,False,None,https://doi.org/10.1021/acsomega.2c03671,Staphylococcus aureus is considered as one of ...
7,"Crystal Structure of 6,7-Dihydro-5a,7a,13,14-t...",10.2116/xraystruct.35.57,2019,semantic_scholar,Jean Guillon; Maria de Fatima Pereira-Rosenfel...,False,None,https://www.jstage.jst.go.jp/article/xraystruc...,"The X-ray crystal structure of 6,7-dihydro-5a,..."
8,The Staphylococcus aureus Methicillin Resistan...,10.1128/mBio.02070-15,2016,semantic_scholar,M. M. Rahman; H. Hunter; S. Prova et al.,False,None,https://mbio.asm.org/content/mbio/7/1/e02070-1...,ABSTRACT The methicillin resistance factor enc...
9,ALS Mutations Disrupt Phase Separation Mediate...,10.1016/j.str.2016.07.007,2016,semantic_scholar,Alexander E. Conicella; Gül H. Zerze; J. Mitta...,False,None,https://doi.org/10.1016/j.str.2016.07.007,


### 3c · Elsevier (ScienceDirect)

In [3]:
async with ElsevierClient() as client:
    els_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"Elsevier returned {len(els_papers)} papers")
papers_to_df(els_papers)

17:14:45 [INFO] httpx – HTTP Request: GET https://api.elsevier.com/content/search/sciencedirect?query=FmtA+structure&count=10&view=COMPLETE "HTTP/1.1 200 OK"


Elsevier returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,Repurposing an Ancient Protein Core Structure:...,10.1016/j.jmb.2019.06.019,2019,elsevier,Vikram Dalal; Pramod Kumar; Pravindra Kumar,False,None,https://api.elsevier.com/content/article/pii/S...,
1,Identification of a <ce:italic>fmtA</ce:italic...,10.1016/S0378-1097(00)00194-4,2000,elsevier,Hitoshi Komatsuzawa; Gil H. Choi; Hidekazu Sug...,False,None,https://api.elsevier.com/content/article/pii/S...,
2,Antibacterial potential of Trichoderma bioacti...,10.1016/j.amolm.2025.100076,2025,elsevier,Gourav Choudhir; Israil; Asimul Islam,False,None,https://api.elsevier.com/content/article/pii/S...,
3,Development of novel PET probes targeting phos...,10.1016/j.nucmedbio.2015.09.008,2016,elsevier,Akira Makino; Takahiro Arai; Hideo Saji,False,None,https://api.elsevier.com/content/article/pii/S...,
4,Analysis of Structure–Function Relationships i...,10.1016/j.jmb.2012.09.017,2012,elsevier,Antony Cougnoux; Lucie Gibold; Richard Bonnet,False,None,https://api.elsevier.com/content/article/pii/S...,
5,In-silico functional and structural annotation...,10.1016/j.jmgm.2022.108262,2022,elsevier,Vishakha Singh; Poonam Dhankhar; Pravindra Kumar,False,None,https://api.elsevier.com/content/article/pii/S...,
6,Diversity of Penicillin-binding Proteins,10.1074/jbc.M706296200,2007,elsevier,Xin Fan; Yuhong Liu; Dasantila Golemi-Kotra,False,None,https://api.elsevier.com/content/article/pii/S...,
7,Effects of different covalent organic framewor...,10.1016/j.jece.2024.114193,2024,elsevier,Yuchen Zhang; Qiao Ma; Xiangke Wang,False,None,https://api.elsevier.com/content/article/pii/S...,
8,Structure and function of prodrug-activating p...,10.1016/j.biochi.2022.07.019,2023,elsevier,José A. Velilla; Grace E. Kenney; Rachelle Gaudet,False,None,https://api.elsevier.com/content/article/pii/S...,
9,Pyramid array evaporators: Synergizing solar e...,10.1016/j.cej.2025.163574,2025,elsevier,Yinlong Li; Yulong Tong; Beibei Wang,False,None,https://api.elsevier.com/content/article/pii/S...,


### 3d · CrossRef

In [6]:
async with CrossRefClient() as client:
    cr_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"CrossRef returned {len(cr_papers)} papers")
papers_to_df(cr_papers)

01:10:38 [INFO] httpx – HTTP Request: GET https://api.crossref.org/works?query=FmtA+structure&rows=10&select=DOI%2Ctitle%2Cauthor%2Cabstract%2Cpublished%2CURL "HTTP/1.1 200 OK"


CrossRef returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,Crystal Structure of FmtA from Staphylococcus ...,10.2210/pdb5zh8/pdb,2019.0,crossref,V. Dalal; P. Kumar; D. Golemi-Kotra et al.,False,None,https://doi.org/10.2210/pdb5zh8/pdb,
1,Structure of FmtA-like protein,10.2210/pdb4gdn/pdb,2012.0,crossref,A. Cougnoux; L. Gibold; J. Delmas et al.,False,None,https://doi.org/10.2210/pdb4gdn/pdb,
2,Quantum Mechanics/Molecular Mechanics Studies ...,10.1021/acs.jcim.2c00057.s001,NaN,crossref,,False,None,https://doi.org/10.1021/acs.jcim.2c00057.s001,
3,Determination of VraR Binding Activity on fmtA...,10.1096/fasebj.27.1_supplement.769.2,2013.0,crossref,Zhifeng Yang; Dasantila Golemi‐Kotra,False,None,https://doi.org/10.1096/fasebj.27.1_supplement...,<jats:p>\n In\n ...
4,Identification of a fmtA-like gene that has si...,10.1016/s0378-1097(00)00194-4,2000.0,crossref,H Komatsuzawa,False,None,https://doi.org/10.1016/s0378-1097(00)00194-4,
5,Faculty Opinions recommendation of The Staphyl...,10.3410/f.726135701.793521683,2016.0,crossref,Yu Luo,False,None,https://doi.org/10.3410/f.726135701.793521683,
6,Dual Roles of FmtA in Staphylococcus aureus Ce...,10.1128/aac.00187-12,2012.0,crossref,Aneela Qamar; Dasantila Golemi-Kotra,False,None,https://doi.org/10.1128/aac.00187-12,<jats:title>ABSTRACT</jats:title>\n <...
7,Structure-Based Identification of Potential Dr...,10.1007/s10930-020-09953-6,2021.0,crossref,Vikram Dalal; Poonam Dhankhar; Vishakha Singh ...,False,None,https://doi.org/10.1007/s10930-020-09953-6,
8,Staphylococcus aureus Methicillin-Resistance F...,10.1371/journal.pone.0043998,2012.0,crossref,Yinglu Zhao; Vidhu Verma; Antoaneta Belcheva e...,False,None,https://doi.org/10.1371/journal.pone.0043998,
9,Forensic mental telehealth assessment (FMTA) i...,10.1016/j.ijlp.2020.101595,2020.0,crossref,Eric Y. Drogin,False,None,https://doi.org/10.1016/j.ijlp.2020.101595,


### 3e · OpenAlex

In [7]:
async with OpenAlexClient() as client:
    oa_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"OpenAlex returned {len(oa_papers)} papers")
papers_to_df(oa_papers)

01:10:40 [INFO] httpx – HTTP Request: GET https://api.openalex.org/works?search=FmtA+structure&per-page=10&select=id%2Cdoi%2Ctitle%2Cauthorships%2Cabstract_inverted_index%2Cpublication_year%2Copen_access%2Cprimary_location "HTTP/1.1 200 OK"


OpenAlex returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,Structure-Based Identification of Potential Dr...,10.1007/s10930-020-09953-6,2021,openalex,Vikram Dalal; Poonam Dhankhar; Vishakha Singh ...,False,None,https://doi.org/10.1007/s10930-020-09953-6,
1,Repurposing an Ancient Protein Core Structure:...,10.1016/j.jmb.2019.06.019,2019,openalex,Vikram Dalal; Pramod Kumar; Gaddy Rakhaminov e...,False,None,https://doi.org/10.1016/j.jmb.2019.06.019,
2,Repurposing an ancient protein core structure:...,10.1107/s010876732108990x,2021,openalex,Vikram L. Dalal; Pramod Kumar; Gaddy Rakhamino...,False,None,https://journals.iucr.org/a/issues/2021/a2/00/...,FmtA is a penicillin-recognizing protein (PRP)...
3,"Characterization of <i>fmtA</i> , a Gene That ...",10.1128/aac.43.9.2121,1999,openalex,Hitoshi Komatsuzawa; Kouji Ohta; Harald Labisc...,False,None,https://aac.asm.org/content/aac/43/9/2121.full...,FmtA is a factor which affects the methicillin...
4,Constructing Stable and Porous Covalent Organi...,10.1002/marc.202100032,2021,openalex,Lipeng Zhai; Diandian Han; Jinhuan Dong et al.,False,None,https://doi.org/10.1002/marc.202100032,Covalent organic frameworks (COF) with periodi...
5,Drug-Repurposing Approach To Combat <i>Staphyl...,10.1021/acsomega.2c03671,2022,openalex,Vishakha Singh; Poonam Dhankhar; Vikram Dalal ...,False,None,https://doi.org/10.1021/acsomega.2c03671,[Image: see text] Staphylococcus aureus is con...
6,Analysis of Structure–Function Relationships i...,10.1016/j.jmb.2012.09.017,2012,openalex,Antony Cougnoux; Lucie Gibold; Frédéric Robin ...,False,None,https://doi.org/10.1016/j.jmb.2012.09.017,
7,In-silico functional and structural annotation...,10.1016/j.jmgm.2022.108262,2022,openalex,Vishakha Singh; Poonam Dhankhar; Vikram Dalal ...,False,None,https://doi.org/10.1016/j.jmgm.2022.108262,
8,Identification of Genes Involved in Polysaccha...,10.1371/journal.pone.0010146,2010,openalex,Blaise R. Boles; Matthew Thoendel; Aleeza J. R...,False,None,https://journals.plos.org/plosone/article/file...,Staphylococcus aureus is a potent biofilm form...
9,Structure of FmtA-like protein,10.2210/pdb4gdn/pdb,2012,openalex,Antony Cougnoux; L. Gibold; Julien Delmas et al.,False,None,https://doi.org/10.2210/pdb4gdn/pdb,


### 3f · Scopus

In [8]:
async with ScopusClient() as client:
    scopus_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"Scopus returned {len(scopus_papers)} papers")
papers_to_df(scopus_papers)

01:10:41 [INFO] httpx – HTTP Request: GET https://api.elsevier.com/content/search/scopus?query=FmtA+structure&count=10&field=prism%3Adoi%2Cdc%3Atitle%2Cdc%3Acreator%2Cauthor%2Cprism%3AcoverDate%2Cdc%3Adescription%2Cprism%3Aurl "HTTP/1.1 200 OK"


Scopus returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,Evaluating the ability of in silico identified...,10.1007/s11030-025-11155-0,2026,scopus,Boggarapu Ganesh; Adrija Banerjee; Lalitha Gur...,False,None,https://doi.org/10.1007/s11030-025-11155-0,
1,Research progress on the inflammatory effects ...,,2026,scopus,Can Zhang; Ping Gui; Qian Xiu Zhao et al.,False,None,https://api.elsevier.com/content/abstract/scop...,
2,Structure-based drug discovery targeting fmhA ...,10.1515/znc-2025-0274,2026,scopus,Nouman Ali; Muhammad Naveed; Muhammad Waseem e...,False,None,https://doi.org/10.1515/znc-2025-0274,
3,Antimicrobial Potential of Bempedoic Acid as a...,10.2174/0122113525391531250604064623,2026,scopus,Ketan Sharma; Anuj Kumar Sharma; Yogesh Murti ...,False,None,https://doi.org/10.2174/0122113525391531250604...,
4,Discovery of Two GSK3β Inhibitors from Sophora...,10.2174/0115734099321878241011104241,2026,scopus,Dabo Pan; Yong Zeng; Dewen Jiang et al.,False,None,https://doi.org/10.2174/0115734099321878241011...,
5,"GC–MS phytochemical screening, molecular docki...",10.1007/s00210-025-04875-5,2026,scopus,Nina Bhagyanath; C. Pushpaveni; Monica Arora e...,False,None,https://doi.org/10.1007/s00210-025-04875-5,
6,TargetGen-RNN model discovers novel antibiotic...,10.1002/inmd.70064,2026,scopus,Shakeel Ahmad Khan; Adnan Shakoor; Sadia Kanwal,False,None,https://doi.org/10.1002/inmd.70064,
7,Methicillin-Resistant Staphylococcus epidermid...,10.2174/0122113525388954250603114510,2026,scopus,Sourav Ghosh; Shelly Singh; Priya Kaushik,False,None,https://doi.org/10.2174/0122113525388954250603...,
8,Studying on the effect and mechanism of four n...,10.1016/j.molstruc.2025.143320,2025,scopus,Guang Hua Tong; Fei Yu Zhang; Hui Xin Zheng et...,False,None,https://doi.org/10.1016/j.molstruc.2025.143320,
9,Novel triazole-based N-Acetyl schiff bases: Sy...,10.1016/j.molstruc.2025.143176,2025,scopus,Hilal Medetalibeyoğlu; Burak Tüzün; Abdurrahma...,False,None,https://doi.org/10.1016/j.molstruc.2025.143176,


### 3g · PubMed / PMC

In [9]:
async with PMCClient() as client:
    pmc_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"PMC returned {len(pmc_papers)} papers")
papers_to_df(pmc_papers)

01:10:42 [INFO] httpx – HTTP Request: GET https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?tool=Pipetly&email=contact%40pipetly.bot&retmode=json&db=pubmed&term=FmtA+structure&retmax=10 "HTTP/1.1 200 OK"
01:10:42 [INFO] httpx – HTTP Request: GET https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?tool=Pipetly&email=contact%40pipetly.bot&retmode=json&db=pubmed&id=36340146%2C35839717%2C35475370%2C34050692%2C33421024%2C31260692%2C28192825%2C23041299%2C22952845%2C10471551 "HTTP/1.1 200 OK"


PMC returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,Drug-Repurposing Approach To Combat Staphyloco...,10.1021/acsomega.2c03671,2022,pmc,Singh V; Dhankhar P; Dalal V et al.,False,None,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9...,
1,In-silico functional and structural annotation...,10.1016/j.jmgm.2022.108262,2022,pmc,Singh V; Dhankhar P; Dalal V et al.,False,None,,
2,Quantum Mechanics/Molecular Mechanics Studies ...,10.1021/acs.jcim.2c00057,2022,pmc,Dalal V; Golemi-Kotra D; Kumar P,False,None,,
3,Constructing Stable and Porous Covalent Organi...,10.1002/marc.202100032,2021,pmc,Zhai L; Han D; Dong J et al.,False,None,,
4,Structure-Based Identification of Potential Dr...,10.1007/s10930-020-09953-6,2021,pmc,Dalal V; Dhankhar P; Singh V et al.,False,None,,
5,Repurposing an Ancient Protein Core Structure:...,10.1016/j.jmb.2019.06.019,2019,pmc,Dalal V; Kumar P; Rakhaminov G et al.,False,None,,
6,Dynamic Alignment Analysis in the Osteoarthrit...,10.1055/s-0037-1598037,2017,pmc,Larrainzar-Garijo R; Murillo-Vizuete D; Garcia...,False,None,,
7,Analysis of structure-function relationships i...,10.1016/j.jmb.2012.09.017,2012,pmc,Cougnoux A; Gibold L; Robin F et al.,False,None,,
8,Staphylococcus aureus methicillin-resistance f...,10.1371/journal.pone.0043998,2012,pmc,Zhao Y; Verma V; Belcheva A et al.,False,None,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3...,
9,"Characterization of fmtA, a gene that modulate...",10.1128/AAC.43.9.2121,1999,pmc,Komatsuzawa H; Ohta K; Labischinski H et al.,False,None,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8...,


### 3h · CORE


In [10]:
async with COREClient() as client:
    core_papers = await client.search(QUERY, MAX_PER_SOURCE)
print(f"CORE returned {len(core_papers)} papers")
papers_to_df(core_papers)

01:10:50 [INFO] httpx – HTTP Request: GET https://api.core.ac.uk/v3/search/works?q=FmtA+structure&limit=10 "HTTP/1.1 301 Moved Permanently"
01:10:59 [INFO] httpx – HTTP Request: GET https://api.core.ac.uk/v3/search/works/?q=FmtA+structure&limit=10 "HTTP/1.1 200 OK"


CORE returned 10 papers


,title,doi,year,source,authors,has_full_text,abstract_only,url,abstract
0,Draft Genome Sequence of Staphylococcus pseudi...,10.17169/refubium-26458,2020,core,"Bäumer, Wolfgang; Eichhorn, Inga; Fulde, Marcu...",False,None,https://core.ac.uk/download/288114168.pdf,"Here, we report the draft genome sequence of S..."
1,The Dimerization Interface in VraR is Essentia...,10.1186/s12866-019-1529-0,2019,core,"Golemi-Kotra, Dasantila; Tajbakhsh, Ghazal",False,None,https://core.ac.uk/download/287660067.pdf,Background\r\nStaphylococcus aureus remains a ...
2,ClbP is a prototype of a peptidase subgroup in...,10.1074/jbc.m111.221960,2011,core,"Baron, O.; Bonnet, R.; Bouchon, B. et al.",False,None,https://core.ac.uk/download/39834780.pdf,The pks genomic island of Escherichia coli enc...
3,Staphylococcus aureus Survives with a Minimal ...,10.1371/journal.ppat.1004891,2015,core,A Antignac; A Bera; A Typas et al.,False,None,https://core.ac.uk/download/42623201.pdf,Many important cellular processes are performe...
4,The Ifo Investment Database,10.3790/schm.133.3.449,2013,core,"Sauer, Stefan; Strobel, Thomas; Wohlrabe, Klaus",False,None,https://www.econstor.eu/dspace/bitstream/10419...,In this paper we present the Ifo Investment Da...
5,How Foucault’s Panopticon Governs Special Educ...,,2015,core,"Angus, Gail; Winslade, John M",False,None,https://core.ac.uk/download/55333130.pdf,Special education laws in California function ...
6,Influence of Dielectric Environment upon Isoto...,10.1021/jacs.9b11988,2019,core,"Porter, Alexander J.; Upfold, Catherine M.; Wi...",False,None,https://core.ac.uk/download/304372029.pdf,Isotope effects depend upon the polarity of th...
7,Probing pattern and dynamics of disulfide brid...,10.1039/c5sc03995a,2016,core,"Bartok, Adam; Fehér, Krisztina; Illyés, Tünde ...",False,None,https://core.ac.uk/download/84046448.pdf,"Anuroctoxin (AnTx), a 35-amino-acid scorpion t..."
8,The running coupling of 8 flavors and 3 colors,10.1007/jhep06(2015)019,2015,core,A Cheng; A Chowdhury; A Coste et al.,False,None,https://core.ac.uk/download/35067163.pdf,We compute the renormalized running coupling o...
9,"The 'Antiretrovirals, Sexual Transmission Risk...",10.1371/journal.pone.0077230,2013,core,Alec Miners; Alison Rodger; Andrew N. Phillips...,False,None,https://core.ac.uk/download/17192885.pdf,Life expectancy for people diagnosed with HIV ...


## 4 · Deduplication

Combines results from all nine section-3 clients, deduplicates by DOI (falling
back to normalised title for records without a DOI), and produces `_pool` — the
unique paper list used by every full-text retrieval cell in section 5.

In [11]:
from collections import defaultdict

# ── 1. All raw results per source ─────────────────────────────────────────────
_source_raw: dict[str, list] = {
    "europe_pmc":       epmc_papers,
    "semantic_scholar": s2_papers,
    "elsevier":         els_papers,
    "crossref":         cr_papers,
    "openalex":         oa_papers,
    "scopus":           scopus_papers,
    "pmc":              pmc_papers,
    "unpaywall":        [],            # no search endpoint
    "core":             core_papers,
}

_pool_raw: list = []
for _papers in _source_raw.values():
    _pool_raw.extend(_papers)

# ── 2. Deduplicate (DOI-first, normalised title fallback) ─────────────────────
_pool_seen: dict[str, "Paper"] = {}   # uid → first-seen Paper object
_uid_sources: dict[str, list[str]] = defaultdict(list)  # uid → [sources]

for _src, _papers in _source_raw.items():
    for _p in _papers:
        _uid = _p.unique_id()
        _uid_sources[_uid].append(_src)
        if _uid not in _pool_seen:
            _pool_seen[_uid] = _p

_pool = list(_pool_seen.values())

# ── 3. Deduplication summary ──────────────────────────────────────────────────
_n_raw   = len(_pool_raw)
_n_uniq  = len(_pool)
_n_dupes = _n_raw - _n_uniq
_shared  = {uid: srcs for uid, srcs in _uid_sources.items() if len(srcs) > 1}

print("=" * 55)
print("  DEDUPLICATION SUMMARY")
print("=" * 55)
print(f"  Raw papers (all sources combined) : {_n_raw:>5}")
print(f"  Unique papers after dedup         : {_n_uniq:>5}")
print(f"  Duplicate records removed         : {_n_dupes:>5}")
print(f"  Papers shared by ≥2 sources       : {len(_shared):>5}")
print()

# ── 4. Per-source breakdown ───────────────────────────────────────────────────
print(f"{'Source':<22} {'Raw':>5}  {'w/ DOI':>7}  {'No DOI':>7}")
print("-" * 46)
for _src, _papers in _source_raw.items():
    _n_doi  = sum(1 for _p in _papers if _p.doi)
    _n_none = len(_papers) - _n_doi
    print(f"  {_src:<20} {len(_papers):>5}  {_n_doi:>7}  {_n_none:>7}")
print("-" * 46)
print(f"  {'TOTAL':<20} {_n_raw:>5}  {sum(1 for _p in _pool_raw if _p.doi):>7}  {sum(1 for _p in _pool_raw if not _p.doi):>7}")
print()

# ── 5. Cross-source overlap ───────────────────────────────────────────────────
if _shared:
    print(f"Cross-source duplicates (sample of up to 5):")
    for _uid, _srcs in list(_shared.items())[:5]:
        print(f"  {_uid[:52]}  ← {', '.join(_srcs)}")
else:
    print("No cross-source duplicates found in this query.")


  DEDUPLICATION SUMMARY
  Raw papers (all sources combined) :    80
  Unique papers after dedup         :    57
  Duplicate records removed         :    23
  Papers shared by ≥2 sources       :    12

Source                   Raw   w/ DOI   No DOI
----------------------------------------------
  europe_pmc              10       10        0
  semantic_scholar        10       10        0
  elsevier                10       10        0
  crossref                10       10        0
  openalex                10       10        0
  scopus                  10        9        1
  pmc                     10       10        0
  unpaywall                0        0        0
  core                    10        9        1
----------------------------------------------
  TOTAL                   80       78        2

Cross-source duplicates (sample of up to 5):
  10.1021/acsomega.2c03671  ← europe_pmc, semantic_scholar, openalex, pmc
  10.1016/j.biochi.2022.07.019  ← europe_pmc, elsevier
  10.1007/s10

In [12]:
# ── Deduplication analytics as a DataFrame ────────────────────────────────────
import pandas as pd

_dedup_rows = []
for _src, _papers in _source_raw.items():
    _with_doi  = sum(1 for _p in _papers if _p.doi)
    _no_doi    = len(_papers) - _with_doi
    # How many of this source's papers are unique vs. seen earlier
    _src_uids  = {(_p.doi.strip().lower() if _p.doi else _p.title.strip().lower())
                  for _p in _papers}
    _unique_contributed = sum(
        1 for uid in _src_uids
        if _uid_sources[uid][0] == _src   # this source was first to see it
    )
    _dedup_rows.append({
        "source":             _src,
        "raw_papers":         len(_papers),
        "with_doi":           _with_doi,
        "no_doi":             _no_doi,
        "unique_contributed": _unique_contributed,
        "duplicated_away":    len(_papers) - _unique_contributed,
    })

_dedup_df = pd.DataFrame(_dedup_rows).set_index("source")
print(f"Pool size after dedup: {len(_pool)} unique papers\n")
_dedup_df

Pool size after dedup: 57 unique papers



,raw_papers,with_doi,no_doi,unique_contributed,duplicated_away
source,,,,,
europe_pmc,10,10,0,10,0
semantic_scholar,10,10,0,9,1
elsevier,10,10,0,6,4
crossref,10,10,0,6,4
openalex,10,10,0,4,6
scopus,10,9,1,10,0
pmc,10,10,0,2,8
unpaywall,0,0,0,0,0
core,10,9,1,10,0


## 5 · Individual Client Full-Text Retrieval

Each subsection runs `fetch_full_text` for **every paper in `_pool`**
(built and deduplicated in section 4) through that one client.
This tests each API's full-text coverage independently of which client
originally discovered a paper.

Results are stored as `{client}_ft_map: dict[str, str]` (unique-id → text) and
merged in **5j** to build `raw_papers` for sections 6 and 7.

In [13]:
# ── 5·0  Verify the deduplicated pool is available ───────────────────────────
# `_pool` and `_pool_seen` are built in section 4.
# Run all section-3 cells AND the section-4 deduplication cells first.
assert "_pool" in dir(), "Run section 4 deduplication cells before section 5."
print(f"Pool ready: {len(_pool)} unique papers — running full-text clients below.")

Pool ready: 57 unique papers — running full-text clients below.


### 5a · Europe PMC

In [14]:
epmc_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with EuropePMCClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            epmc_ft_map[_uid] = _ft

print(f"EuropePMC: full text retrieved for {len(epmc_ft_map)} / {len(_pool)} papers")
for _uid, _ft_obj in list(epmc_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


01:11:01 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/PMC12505958/fullTextXML "HTTP/1.1 200 OK"
01:11:02 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/PMC12229560/fullTextXML "HTTP/1.1 200 OK"
01:11:04 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/PMC11822514/fullTextXML "HTTP/1.1 200 OK"
01:11:05 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/PMC12393781/fullTextXML "HTTP/1.1 200 OK"
01:11:08 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/PMC12094522/fullTextXML "HTTP/1.1 200 OK"
01:11:09 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/PMC9631409/fullTextXML "HTTP/1.1 200 OK"
01:11:09 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europepmc/webservices/rest/PMC12594824/fullTextXML "HTTP/1.1 200 OK"
01:11:10 [INFO] httpx – HTTP Request: GET https://www.ebi.ac.uk/europe

EuropePMC: full text retrieved for 17 / 57 papers
  10.1128/mbio.02337-25  -> 229,557 chars
  10.1038/s44259-025-00131-1  -> 238,637 chars
  10.1021/acsomega.4c06520  -> 207,019 chars


### 5b · Semantic Scholar

In [15]:
s2_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with SemanticScholarClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            s2_ft_map[_uid] = _ft

print(f"Semantic Scholar: full text retrieved for {len(s2_ft_map)} / {len(_pool)} papers")
for _uid, _ft_obj in list(s2_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


01:11:28 [INFO] httpx – HTTP Request: GET https://api.semanticscholar.org/graph/v1/paper/DOI:10.1128/mbio.02337-25?fields=openAccessPdf "HTTP/1.1 200 OK"
01:11:28 [INFO] httpx – HTTP Request: GET https://api.semanticscholar.org/graph/v1/paper/DOI:10.1038/s44259-025-00131-1?fields=openAccessPdf "HTTP/1.1 200 OK"
01:11:29 [INFO] httpx – HTTP Request: GET https://api.semanticscholar.org/graph/v1/paper/DOI:10.1021/acsomega.4c06520?fields=openAccessPdf "HTTP/1.1 200 OK"
01:11:29 [INFO] httpx – HTTP Request: GET https://doi.org/10.1021/acsomega.4c06520 "HTTP/1.1 302 Found"
01:11:30 [INFO] httpx – HTTP Request: GET https://pubs.acs.org/doi/10.1021/acsomega.4c06520 "HTTP/1.1 403 Forbidden"
01:11:30 [WARNING] api_clients.semantic_scholar – SemanticScholar full-text fetch failed: Client error '403 Forbidden' for url 'https://pubs.acs.org/doi/10.1021/acsomega.4c06520'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
01:11:31 [INFO] httpx – HTTP Request: GET

Semantic Scholar: full text retrieved for 13 / 57 papers
  10.1016/j.jpha.2024.01.006  -> 3,736 chars
  10.1016/j.biochi.2022.07.019  -> 3,560 chars
  10.2116/xraystruct.35.57  -> 214,876 chars


### 5c · Elsevier (ScienceDirect)

In [16]:
els_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with ElsevierClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            els_ft_map[_uid] = _ft

print(f"Elsevier: full text retrieved for {len(els_ft_map)} / {len(_pool)} papers")
if not els_ft_map:
    print("  (0 results is normal if ELSEVIER_API_KEY is not set or articles are not licensed)")
for _uid, _ft_obj in list(els_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


01:16:02 [INFO] httpx – HTTP Request: GET https://api.elsevier.com/content/article/doi/10.1128/mbio.02337-25 "HTTP/1.1 404 Not Found"
01:16:02 [WARNING] api_clients.elsevier – Elsevier full-text fetch failed for 10.1128/mbio.02337-25: Client error '404 Not Found' for url 'https://api.elsevier.com/content/article/doi/10.1128/mbio.02337-25'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
01:16:02 [INFO] httpx – HTTP Request: GET https://api.elsevier.com/content/article/doi/10.1038/s44259-025-00131-1 "HTTP/1.1 404 Not Found"
01:16:02 [WARNING] api_clients.elsevier – Elsevier full-text fetch failed for 10.1038/s44259-025-00131-1: Client error '404 Not Found' for url 'https://api.elsevier.com/content/article/doi/10.1038/s44259-025-00131-1'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
01:16:02 [INFO] httpx – HTTP Request: GET https://api.elsevier.com/content/article/doi/10.1021/acsomega.4c06520 "HTTP/1.1 404

Elsevier: full text retrieved for 16 / 57 papers
  10.1016/j.jpha.2024.01.006  -> 23,356 chars
  10.1016/j.biochi.2022.07.019  -> 101,965 chars
  10.1016/j.jmb.2019.06.019  -> 107,723 chars


### 5d · PubMed / PMC

In [17]:
pmc_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with PMCClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            pmc_ft_map[_uid] = _ft

print(f"PMC: full text retrieved for {len(pmc_ft_map)} / {len(_pool)} papers")
if not pmc_ft_map:
    print("  (0 results means no paper in the pool has a PMC open-access record)")
for _uid, _ft_obj in list(pmc_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


01:16:26 [INFO] httpx – HTTP Request: GET https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?tool=Pipetly&email=contact%40pipetly.bot&retmode=json&db=pmc&term=10.1128%2Fmbio.02337-25%5BDOI%5D&retmax=1 "HTTP/1.1 200 OK"
01:16:26 [INFO] httpx – HTTP Request: GET https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?tool=Pipetly&email=contact%40pipetly.bot&db=pmc&id=12505958&rettype=full&retmode=xml "HTTP/1.1 200 OK"
01:16:27 [INFO] httpx – HTTP Request: GET https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?tool=Pipetly&email=contact%40pipetly.bot&retmode=json&db=pmc&term=10.1038%2Fs44259-025-00131-1%5BDOI%5D&retmax=1 "HTTP/1.1 200 OK"
01:16:27 [INFO] httpx – HTTP Request: GET https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?tool=Pipetly&email=contact%40pipetly.bot&db=pmc&id=12229560&rettype=full&retmode=xml "HTTP/1.1 200 OK"
01:16:28 [INFO] httpx – HTTP Request: GET https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?tool=Pipetly&email=contact%40pipet

PMC: full text retrieved for 22 / 57 papers
  10.1128/mbio.02337-25  -> 226,017 chars
  10.1038/s44259-025-00131-1  -> 234,162 chars
  10.1021/acsomega.4c06520  -> 208,443 chars


### 5e · Unpaywall

No search endpoint — resolves any DOI to the best open-access PDF.  
Requires `UNPAYWALL_EMAIL` in `.env`.

In [18]:
upw_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with UnpaywallClient() as client:
    for _p in _pool:
        if not _p.doi:
            continue   # Unpaywall requires a DOI
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            upw_ft_map[_uid] = _ft

_doi_count = sum(1 for _p in _pool if _p.doi)
print(f"Unpaywall: full text retrieved for {len(upw_ft_map)} / {_doi_count} papers with a DOI")
if not upw_ft_map:
    print("  (0 results is normal for subscription-only papers or if UNPAYWALL_EMAIL is not set)")
for _uid, _ft_obj in list(upw_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


01:17:14 [INFO] httpx – HTTP Request: GET https://api.unpaywall.org/v2/10.1128/mbio.02337-25?email=patatatuyyo%40gmail.com "HTTP/1.1 200 OK"
01:17:14 [INFO] httpx – HTTP Request: GET https://api.unpaywall.org/v2/10.1038/s44259-025-00131-1?email=patatatuyyo%40gmail.com "HTTP/1.1 200 OK"
01:17:14 [INFO] httpx – HTTP Request: GET https://api.unpaywall.org/v2/10.1021/acsomega.4c06520?email=patatatuyyo%40gmail.com "HTTP/1.1 200 OK"
01:17:14 [INFO] httpx – HTTP Request: GET https://api.unpaywall.org/v2/10.3390/applmicrobiol5010032?email=patatatuyyo%40gmail.com "HTTP/1.1 200 OK"
01:17:15 [INFO] httpx – HTTP Request: GET https://www.mdpi.com/2673-8007/5/1/32/pdf?version=1742467354 "HTTP/1.1 403 Forbidden"
01:17:15 [WARNING] api_clients.unpaywall – Unpaywall PDF download failed for 10.3390/applmicrobiol5010032 (https://www.mdpi.com/2673-8007/5/1/32/pdf?version=1742467354): Client error '403 Forbidden' for url 'https://www.mdpi.com/2673-8007/5/1/32/pdf?version=1742467354'
For more information ch

Unpaywall: full text retrieved for 11 / 55 papers with a DOI
  10.1038/s41598-025-24255-6  -> 4,303,156 chars
  10.1016/j.jpha.2024.01.006  -> 2,424 chars
  10.1016/j.biochi.2022.07.019  -> 2,424 chars


### 5f · CORE

Requires `CORE_API_KEY` in `.env` for authenticated access and higher rate limits.

In [19]:
core_ft_map: dict[str, FullText] = {}   # uid -> FullText

async with COREClient() as client:
    for _p in _pool:
        _ft = await client.fetch_full_text(_p)
        if _ft and _ft.content and _ft.content.strip():
            _uid = _p.unique_id()
            core_ft_map[_uid] = _ft

print(f"CORE: full text retrieved for {len(core_ft_map)} / {len(_pool)} papers")
if not core_ft_map:
    print("  (0 results is normal if CORE_API_KEY is not set or articles are not in OA corpus)")
for _uid, _ft_obj in list(core_ft_map.items())[:3]:
    print(f"  {_uid[:60]}  -> {len(_ft_obj.content):,} chars")


01:31:52 [INFO] httpx – HTTP Request: GET https://api.core.ac.uk/v3/works/doi:10.1128/mbio.02337-25 "HTTP/1.1 404 Not Found"
01:31:52 [WARNING] api_clients.core – CORE DOI lookup failed for 10.1128/mbio.02337-25: Client error '404 Not Found' for url 'https://api.core.ac.uk/v3/works/doi:10.1128/mbio.02337-25'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
01:31:54 [INFO] httpx – HTTP Request: GET https://api.core.ac.uk/v3/works/doi:10.1038/s44259-025-00131-1 "HTTP/1.1 404 Not Found"
01:31:54 [WARNING] api_clients.core – CORE DOI lookup failed for 10.1038/s44259-025-00131-1: Client error '404 Not Found' for url 'https://api.core.ac.uk/v3/works/doi:10.1038/s44259-025-00131-1'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
01:31:58 [INFO] httpx – HTTP Request: GET https://api.core.ac.uk/v3/works/doi:10.1021/acsomega.4c06520 "HTTP/1.1 404 Not Found"
01:31:58 [WARNING] api_clients.core – CORE DOI lookup faile

CORE: full text retrieved for 6 / 57 papers
  10.17169/refubium-26458  -> 170,576 chars
  10.1371/journal.ppat.1004891  -> 2,327,104 chars
  10.3790/schm.133.3.449  -> 1,153,008 chars


### 5j · Consolidate — attach best full text & build `raw_papers`

Merges the candidate pool with the per-client full-text maps from 5a–5f.
For each unique paper the **first** client that returned text wins (priority order:
EuropePMC → PMC → Elsevier → Semantic Scholar → Unpaywall → CORE).
The resulting `raw_papers` list is used for sections 6 and 7.

In [20]:
from models.paper import FullText, FullTextFormat

# ── Full-text maps from sections 5a–5f (uid -> FullText) ──────────────────────
# Priority: first map that has a hit for a given paper wins.
# Order must match FullTextRetriever._CLIENT_ORDER in processors/full_text_retriever.py
_ft_maps_ordered = [
    ("EuropePMC",        epmc_ft_map),
    ("PMC",              pmc_ft_map),
    ("Elsevier",         els_ft_map),
    ("Semantic Scholar", s2_ft_map),
    ("Unpaywall",        upw_ft_map),
    ("CORE",             core_ft_map),
]

def _attach_ft(paper: "Paper", ft: FullText) -> None:
    """Attach a FullText object to a paper."""
    paper.full_text = ft

# raw_papers = deduplicated pool (already built above as _pool / _pool_seen)
raw_papers = list(_pool_seen.values())

attached = 0
source_counts: dict[str, int] = {}
for _p in raw_papers:
    _uid = _p.unique_id()
    for _source, _ft_map in _ft_maps_ordered:
        if _uid in _ft_map:
            _ft_obj = _ft_map[_uid]
            _attach_ft(_p, _ft_obj)
            _p.ft_retrieved_by = {"EuropePMC": "europe_pmc", "PMC": "pmc", 
                                   "Elsevier": "elsevier",
                                   "Semantic Scholar": "semantic_scholar",
                                   "Unpaywall": "unpaywall", "CORE": "core"}.get(_source, _source.lower())
            source_counts[_source] = source_counts.get(_source, 0) + 1
            attached += 1
            break   # first hit wins

print(f"Full text attached to {attached} / {len(raw_papers)} papers")
print("\nBreakdown by supplying client:")
for src, cnt in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"  {src:<22} {cnt:>4} papers")

Full text attached to 38 / 57 papers

Breakdown by supplying client:
  EuropePMC                17 papers
  Elsevier                 11 papers
  PMC                       5 papers
  Semantic Scholar          3 papers
  CORE                      2 papers


## 6 · Parse & Display Results

Convert `raw_papers` (built in section 5j) to a tidy DataFrame and display it.

In [21]:
df = papers_to_df(raw_papers)

# Display settings
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 100)

print(f"Shape: {df.shape[0]} papers × {df.shape[1]} columns\n")
df[["source", "title", "doi", "year", "has_full_text", "abstract_only"]]

Shape: 57 papers × 9 columns



,source,title,doi,year,has_full_text,abstract_only
0,europe_pmc,The &lt;i&gt;Staphylococcus aureus&lt;/i&gt; esterase FmtA is essential for ...,10.1128/mbio.02337-25,2025.0,True,False
1,europe_pmc,Enhanced resistance of metal sequestering agents by reconfiguration of the S...,10.1038/s44259-025-00131-1,2025.0,True,False
2,europe_pmc,"Structural, CSD, Molecular Docking, Molecular Dynamics, and Hirshfeld Surfac...",10.1021/acsomega.4c06520,2025.0,True,False
3,europe_pmc,Gallium Resistance in &lt;i&gt;Staphylococcus aureus&lt;/i&gt;: Polymorphism...,10.3390/applmicrobiol5010032,2025.0,True,False
4,europe_pmc,Clinical Isolate of &lt;i&gt;Candida tropicalis&lt;/i&gt; from a Patient in ...,10.4236/ojmm.2025.151002,2025.0,True,False
5,europe_pmc,Drug-Repurposing Approach To Combat <i>Staphylococcus aureus</i>: Biomolecul...,10.1021/acsomega.2c03671,2022.0,True,False
6,europe_pmc,Exploring the inhibition mechanisms of momordin Ic on S. aureus serine/threo...,10.1038/s41598-025-24255-6,2025.0,True,False
7,europe_pmc,Discovery of potent anti-MRSA components from <i>Dalbergia odorifera</i> thr...,10.1016/j.jpha.2024.01.006,2024.0,True,False
8,europe_pmc,Effect of Heat Input on Microstructural Evolution and Impact Toughness of th...,10.3390/ma18051148,2025.0,True,False
9,europe_pmc,Structure and function of prodrug-activating peptidases.,10.1016/j.biochi.2022.07.019,2023.0,True,False


## 7 · Analysis

Counts by source, full-text availability, year distribution, DOI coverage,
and a per-source full-text retrieval summary.

In [22]:
# ── 7a · Papers per source & full-text availability ──────────────────────────
print("=== Papers per source ===")
print(df.groupby("source").size().sort_values(ascending=False).to_string())

print("\n=== Full-text availability ===")
print(df.groupby("source")["has_full_text"].value_counts().to_string())

print("\n=== Abstract-only papers ===")
print(df["abstract_only"].value_counts().to_string())

print("\n=== Papers with a DOI ===")
has_doi = df["doi"].str.strip().astype(bool)
print(f"  With DOI    : {has_doi.sum()}")
print(f"  Without DOI : {(~has_doi).sum()}")

=== Papers per source ===
source
core                10
europe_pmc          10
scopus              10
semantic_scholar     9
elsevier             6
crossref             6
openalex             4
pmc                  2

=== Full-text availability ===
source            has_full_text
core              True              9
                  False             1
crossref          False             3
                  True              3
elsevier          True              6
europe_pmc        True             10
openalex          False             2
                  True              2
pmc               False             2
scopus            False             8
                  True              2
semantic_scholar  True              6
                  False             3

=== Abstract-only papers ===
abstract_only
False    38

=== Papers with a DOI ===
  With DOI    : 55
  Without DOI : 2


In [23]:
# ── 7b · Year distribution & unique DOI count ────────────────────────────────
print("=== Publication year distribution ===")
year_counts = df["year"].dropna().astype(int).value_counts().sort_index(ascending=False)
print(year_counts.to_string())

# ── Deduplicated DOI count (simulating FilterPipeline step 1) ────────────────
print("\n=== Unique DOIs (dedup preview) ===")
unique_dois = df[df["doi"].str.strip().astype(bool)]["doi"].nunique()
print(f"  Unique DOIs : {unique_dois} / {has_doi.sum()} total DOI papers")

=== Publication year distribution ===
year
2026     8
2025    11
2024     2
2023     1
2022     3
2021     3
2020     2
2019     5
2017     1
2016     5
2015     3
2013     3
2012     4
2011     1
2010     1
2007     1
2000     1
1999     1

=== Unique DOIs (dedup preview) ===
  Unique DOIs : 55 / 55 total DOI papers


### 7c · Full-Text Retrieval Success per Client

Shows which clients were able to return full text in section 5.

In [24]:
_ft_summary = {
    "EuropePMC":        epmc_ft_map,
    "PMC":              pmc_ft_map,
    "Elsevier":         els_ft_map,
    "Semantic Scholar": s2_ft_map,
    "Unpaywall":        upw_ft_map,
    "CORE":             core_ft_map,
}

_pool_size = len(_pool)
print("=== Full-Text Retrieval Summary (all clients vs. full pool) ===")
print(f"{'Client':<22} {'Papers with FT':>15}  {'Coverage':>10}")
print("─" * 52)
for name, ft_map in _ft_summary.items():
    n = len(ft_map)
    pct = n / _pool_size * 100 if _pool_size else 0
    bar = "█" * int(pct / 5)
    print(f"{name:<22} {n:>8} / {_pool_size:<5}  {pct:5.1f}%  {bar}")

# Union coverage: how many papers got full text from at least one client
_covered = set()
for ft_map in _ft_summary.values():
    _covered |= set(ft_map.keys())
print(f"\nUnion coverage : {len(_covered)} / {_pool_size} papers "
      f"({len(_covered)/max(_pool_size,1)*100:.1f}%) have full text from ≥1 client")

=== Full-Text Retrieval Summary (all clients vs. full pool) ===
Client                  Papers with FT    Coverage
────────────────────────────────────────────────────
EuropePMC                    17 / 57      29.8%  █████
PMC                          22 / 57      38.6%  ███████
Elsevier                     16 / 57      28.1%  █████
Semantic Scholar             13 / 57      22.8%  ████
Unpaywall                    11 / 57      19.3%  ███
CORE                          6 / 57      10.5%  ██

Union coverage : 38 / 57 papers (66.7%) have full text from ≥1 client


### 7d · Source Overlap — DOI Coverage per Client

Shows how many unique DOIs each client contributed and how many were shared
across multiple sources (indicating duplicates that were collapsed).

In [25]:
from collections import defaultdict

_source_raw = {
    "europe_pmc":       epmc_papers,
    "semantic_scholar": s2_papers,
    "elsevier":         els_papers,
    "crossref":         cr_papers,
    "openalex":         oa_papers,
    "scopus":           scopus_papers,
    "pmc":              pmc_papers,
    "unpaywall":        [],            # no search results
    "core":             core_papers,
}

# Build DOI → sources mapping from raw search results
_doi_sources: dict[str, list[str]] = defaultdict(list)
for _src, _papers in _source_raw.items():
    for _p in _papers:
        if _p.doi:
            _doi_sources[_p.doi.strip().lower()].append(_src)

# Per-source stats
print("=== Raw search results per source ===")
print(f"{'Source':<22} {'Papers':>8} {'w/ DOI':>8} {'Unique DOIs':>12}")
print("─" * 54)
for _src, _papers in _source_raw.items():
    n_doi = sum(1 for _p in _papers if _p.doi)
    unique = len({_p.doi.strip().lower() for _p in _papers if _p.doi})
    print(f"{_src:<22} {len(_papers):>8} {n_doi:>8} {unique:>12}")

# Cross-source overlap
_shared = {doi: srcs for doi, srcs in _doi_sources.items() if len(srcs) > 1}
print(f"\n=== Cross-source DOI overlap ===")
print(f"DOIs found by only 1 source  : {sum(1 for s in _doi_sources.values() if len(s)==1)}")
print(f"DOIs shared by 2+ sources    : {len(_shared)}")
if _shared:
    print("\nShared DOIs (sample):")
    for doi, srcs in list(_shared.items())[:5]:
        print(f"  {doi[:50]}  ← {', '.join(srcs)}")

=== Raw search results per source ===
Source                   Papers   w/ DOI  Unique DOIs
──────────────────────────────────────────────────────
europe_pmc                   10       10           10
semantic_scholar             10       10           10
elsevier                     10       10           10
crossref                     10       10           10
openalex                     10       10           10
scopus                       10        9            9
pmc                          10       10           10
unpaywall                     0        0            0
core                         10        9            9

=== Cross-source DOI overlap ===
DOIs found by only 1 source  : 43
DOIs shared by 2+ sources    : 12

Shared DOIs (sample):
  10.1021/acsomega.2c03671  ← europe_pmc, semantic_scholar, openalex, pmc
  10.1016/j.biochi.2022.07.019  ← europe_pmc, elsevier
  10.1007/s10930-020-09953-6  ← semantic_scholar, crossref, openalex, pmc
  10.1016/j.jmb.2019.06.019  ← semantic